In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim

In [2]:
spark = SparkSession.builder \
    .appName("EinKaufPark Bronze Layer") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/19 23:13:27 WARN Utils: Your hostname, DiwashPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/19 23:13:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 23:13:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
bronze_path = os.path.join(os.getcwd(), "data", "bronze", "transactions")

bronze_df = spark.read.parquet(bronze_path)

In [4]:
silver_df = bronze_df

In [5]:
silver_df.count()

47020

In [8]:
silver_df.filter(col("ship_date") < col("order_date")).show(5)

+------------------+--------------------+----------------+----------------+-------------+--------------------+----------+----------+--------------+-------------+------------+----------+----------+--------------+-----------------+-----------+-----------------+------------------+------------------+----------------+-----------+------------+------+-----------------+---------------+------------+---------------------+--------------+-----------+----------+--------------------+-------------------+--------------------+------------+----------------+----------------+--------+--------------+------------+--------------------+---------------+------------+----------+-------------+---------------+--------------------+--------------------+-------------+----------------+--------+-----------+
|   pos_terminal_id|      transaction_id|       basket_id|        batch_id|source_system|         record_hash|order_date| ship_date|ingestion_date|sales_channel|order_status|  store_id|store_city|store_district|stor

In [10]:
silver_df = silver_df.withColumn("ship_date", when(col("ship_date") < col("order_date"), None).otherwise(col("ship_date")))

In [11]:
silver_df.filter(col("quantity") <= 0).show(5)

+---------------+--------------+---------+--------+-------------+-----------+----------+---------+--------------+-------------+------------+--------+----------+--------------+-----------------+----------+------------+------------------+------------------+----------------+-----------+------------+------+-----------------+---------------+------------+---------------------+--------------+-----------+----------+------------+----------------+-------------------+------------+----------------+-----+--------+--------------+------------+--------------------+---------------+------------+----------+-------------+---------------+-----------------+------------+-------------+----------------+--------+-----------+
|pos_terminal_id|transaction_id|basket_id|batch_id|source_system|record_hash|order_date|ship_date|ingestion_date|sales_channel|order_status|store_id|store_city|store_district|store_postal_code|store_area|store_region|store_country_code|store_country_name|store_size_class|customer_id|custome

In [13]:
silver_df.filter(col("unit_price_eur") <= 0).show(5)

+---------------+--------------+---------+--------+-------------+-----------+----------+---------+--------------+-------------+------------+--------+----------+--------------+-----------------+----------+------------+------------------+------------------+----------------+-----------+------------+------+-----------------+---------------+------------+---------------------+--------------+-----------+----------+------------+----------------+-------------------+------------+----------------+-----+--------+--------------+------------+--------------------+---------------+------------+----------+-------------+---------------+-----------------+------------+-------------+----------------+--------+-----------+
|pos_terminal_id|transaction_id|basket_id|batch_id|source_system|record_hash|order_date|ship_date|ingestion_date|sales_channel|order_status|store_id|store_city|store_district|store_postal_code|store_area|store_region|store_country_code|store_country_name|store_size_class|customer_id|custome

In [17]:
silver_df.filter(col("discount_pct") >= 100).show(5)

+---------------+--------------+---------+--------+-------------+-----------+----------+---------+--------------+-------------+------------+--------+----------+--------------+-----------------+----------+------------+------------------+------------------+----------------+-----------+------------+------+-----------------+---------------+------------+---------------------+--------------+-----------+----------+------------+----------------+-------------------+------------+----------------+-----+--------+--------------+------------+--------------------+---------------+------------+----------+-------------+---------------+-----------------+------------+-------------+----------------+--------+-----------+
|pos_terminal_id|transaction_id|basket_id|batch_id|source_system|record_hash|order_date|ship_date|ingestion_date|sales_channel|order_status|store_id|store_city|store_district|store_postal_code|store_area|store_region|store_country_code|store_country_name|store_size_class|customer_id|custome